# Chapter 4: Nonparametric Filters
## Particle Filter — The Swarm of Plutos

Key demo: EKF fails with symmetric environment (multimodal belief).
Particle filter succeeds by maintaining multiple hypotheses.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import matplotlib.pyplot as plt

from pluto_filters.pluto_filters.particle_filters.particle_filter import (
    AugmentedMCL, Particle
)
from pluto_filters.pluto_filters.kalman_filters.ekf import LANDMARKS, normalize_angle

In [ ]:
np.random.seed(7)

# Symmetric scenario: robot at (1, 0, 0), makes 2 observations
# to ambiguous landmarks (same distance to 2 different landmarks)
TRUE_POSE = np.array([1.0, 0.0, 0.0])
mcl = AugmentedMCL(n_particles=500, map_bounds=(-5, 5, -5, 5))

def true_measurement(landmark_id, pose=TRUE_POSE):
    lx, ly = LANDMARKS[landmark_id]
    dx, dy = lx - pose[0], ly - pose[1]
    r = np.sqrt(dx**2 + dy**2) + np.random.normal(0, 0.2)
    phi = normalize_angle(np.arctan2(dy, dx) - pose[2]) + np.random.normal(0, 0.05)
    return r, phi

snapshots = []

def snapshot(label):
    xs = [p.x for p in mcl.particles]
    ys = [p.y for p in mcl.particles]
    ws = [p.weight for p in mcl.particles]
    snapshots.append((xs, ys, ws, label))

snapshot('Prior (uniform)')

# Observe landmark 0
r0, phi0 = true_measurement(0)
mcl.update(0, r0, phi0)
mcl.resample()
snapshot(f'After obs lm0 (r={r0:.1f}m)')

# Move slightly forward
mcl.predict(0.5, 0.0, 0.5)
snapshot('After move +0.5m')

# Observe landmark 2
r2, phi2 = true_measurement(2)
mcl.update(2, r2, phi2)
mcl.resample()
snapshot(f'After obs lm2 → converging')

x, y, theta = mcl.mean_pose()
print(f'Mean pose: ({x:.3f}, {y:.3f}, θ={np.degrees(theta):.1f}°)')
print(f'True pose: ({TRUE_POSE[0]:.3f}, {TRUE_POSE[1]:.3f}, 0.0°)')
print(f'ESS: {mcl.effective_n():.1f} / 500')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Particle Filter — Swarm of Plutos', fontsize=14)

for ax, (xs, ys, ws, label) in zip(axes.flat, snapshots):
    ws_arr = np.array(ws)
    ws_norm = ws_arr / (ws_arr.max() + 1e-300)
    sc = ax.scatter(xs, ys, c=ws_norm, cmap='Blues', s=8, alpha=0.7, vmin=0, vmax=1)
    for lm_id, (lx, ly) in LANDMARKS.items():
        ax.plot(lx, ly, 'r^', markersize=8)
    ax.plot(*TRUE_POSE[:2], 'g*', markersize=14, zorder=10, label='True pos')
    ax.set_xlim(-5.5, 5.5)
    ax.set_ylim(-5.5, 5.5)
    ax.set_title(label)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('ch04_particle_filter.png', dpi=120)
plt.show()

## Exercises
1. Kidnap the robot: after convergence, reset `mcl._particles` to a wrong location. Does Augmented MCL recover?
2. Reduce N_PARTICLES to 50. When does particle deprivation occur?
3. Compare systematic resampling vs multinomial resampling — which has lower variance?